# Dynamic-Scaled Momentum for S&P 500 Stocks

The strategy ranks S&P 500 stocks by volatility-scaled momentum and models execution at the next session’s open. Set DMOM_UNIVERSE_CSV to a dated S&P 500 constituent snapshot with symbol and market_cap columns. Market capitalizations are in U.S. dollars. The constituent and market-cap screen remains fixed, as in the source implementation. Use a snapshot dated at or before the backtest start. Prices come from Yahoo Finance.


In [ ]:
"""
Dynamic-Scaled Momentum (dMOM) Trading Strategy
================================================

This strategy combines momentum signals with volatility scaling to dynamically
adjust position sizes based on both expected returns and risk.

Key Features:
- Momentum calculation using configurable lookback periods
- Volatility-based position scaling
- Risk-adjusted expected returns
- Realistic transaction costs
- Proper trading flow: Signal at Close[t] → Execute at Open[t+1]
- No look-ahead bias

Author: Trading Strategy Implementation
Date: 2025
"""

import pandas as pd
import numpy as np
import os
from _market_data import download_ohlcv
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

class StrategyConfig:
    """
    Centralized configuration for the Dynamic-Scaled Momentum strategy.
    All parameters are easily adjustable here.
    """
    # API Configuration

    # Backtest Period
    START_DATE = "2019-01-01"  # 5 years of data
    END_DATE = "2024-12-31"

    # Strategy Parameters
    MOMENTUM_LOOKBACK = 126  # Number of trading days for momentum calculation (approx 6 months)
    VOLATILITY_LOOKBACK = 21  # Number of trading days for volatility calculation (approx 1 month)
    REBALANCE_FREQUENCY = 21  # Rebalance every ~1 month (21 trading days)

    # Position Sizing Parameters
    TARGET_VOLATILITY = 0.15  # Target annual volatility (15%)
    MAX_LEVERAGE = 1.5  # Maximum leverage allowed
    MIN_WEIGHT = 0.0  # Minimum weight per position
    MAX_WEIGHT = 0.3  # Maximum weight per position (concentration limit)

    # Risk Management
    STOP_LOSS_ENABLED = False  # Enable/disable stop loss
    STOP_LOSS_THRESHOLD = -0.15  # Stop loss at -15% from entry

    # Transaction Costs
    COMMISSION_RATE = 0.001  # 10 basis points per trade (0.1%)
    SLIPPAGE_RATE = 0.0005  # 5 basis points slippage (0.05%)

    # Portfolio Parameters
    TOP_N_STOCKS = 20  # Number of stocks to hold in portfolio
    MIN_MOMENTUM_THRESHOLD = 0.0  # Minimum momentum to consider (filter weak signals)

    # Benchmark
    BENCHMARK_TICKER = "URTH"  # iShares MSCI World ETF

    # Universe Selection
    MIN_MARKET_CAP = 1e9  # Minimum market cap in USD (1 billion)
    MIN_TRADING_DAYS = 250  # Minimum number of trading days required


# ============================================================================
# DATA ACQUISITION MODULE
# ============================================================================

class YahooDataFetcher:
    """
    Handles all data acquisition from Financial Modeling Prep API.
    No synthetic data or forward filling - only real historical data.
    """

    def __init__(self):
        filename = os.environ.get("DMOM_UNIVERSE_CSV")
        if not filename:
            raise ValueError("Set DMOM_UNIVERSE_CSV to a dated S&P 500 snapshot.")
        frame = pd.read_csv(filename, dtype={"symbol": str})
        if not {"symbol", "market_cap"}.issubset(frame.columns):
            raise ValueError("The snapshot must have symbol and market_cap columns.")
        frame = frame.dropna(subset=["symbol", "market_cap"]).copy()
        symbols = frame["symbol"].str.strip().str.upper().str.replace(".", "-", regex=False)
        market_caps = pd.to_numeric(frame["market_cap"], errors="raise")
        self.market_caps = dict(zip(symbols, market_caps))

    def get_sp500_tickers(self) -> List[str]:
        return [symbol for symbol in self.market_caps if symbol]

    def get_historical_prices(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        try:
            return download_ohlcv(ticker, start_date, end_date)[["Open", "High", "Low", "Close", "Volume"]]
        except RuntimeError:
            return pd.DataFrame()

    def get_market_cap(self, ticker: str) -> float:
        return self.market_caps.get(ticker, 0.0)


# ============================================================================
# MOMENTUM CALCULATION MODULE
# ============================================================================

class MomentumCalculator:
    """
    Calculates momentum indicators ensuring no look-ahead bias.
    All calculations use only data available at time T for decision at T+1.
    """

    @staticmethod
    def calculate_momentum(prices: pd.Series, lookback: int) -> pd.Series:
        """
        Calculate simple momentum as percentage return over lookback period.

        Formula: (Price[t] - Price[t-lookback]) / Price[t-lookback]

        This is calculated at Close[t], available for decision at Close[t],
        executed at Open[t+1].
        """
        # Shift to ensure we're using t-1 data for time t decision
        # This prevents look-ahead bias
        momentum = prices.pct_change(periods=lookback)
        return momentum

    @staticmethod
    def calculate_risk_adjusted_momentum(prices: pd.Series,
                                        momentum_lookback: int,
                                        vol_lookback: int) -> pd.Series:
        """
        Calculate risk-adjusted momentum (momentum / volatility).

        This scales momentum by realized volatility to get a Sharpe-like measure.
        Higher values indicate better risk-adjusted performance.
        """
        # Calculate raw momentum
        momentum = MomentumCalculator.calculate_momentum(prices, momentum_lookback)

        # Calculate rolling volatility (annualized)
        # Using log returns for volatility calculation (more accurate)
        log_returns = np.log(prices / prices.shift(1))
        volatility = log_returns.rolling(window=vol_lookback).std() * np.sqrt(252)

        # Risk-adjusted momentum = momentum / volatility
        # Higher volatility reduces the momentum score
        risk_adj_momentum = momentum / (volatility + 1e-6)  # Add small epsilon to avoid division by zero

        return risk_adj_momentum


# ============================================================================
# POSITION SIZING MODULE
# ============================================================================

class VolatilityScaler:
    """
    Implements volatility-based position sizing.
    Core concept: Scale positions inversely with volatility to target constant risk.
    """

    @staticmethod
    def calculate_volatility(returns: pd.Series, lookback: int) -> pd.Series:
        """
        Calculate rolling volatility (annualized).

        Uses standard deviation of returns over lookback period,
        annualized by multiplying by sqrt(252).
        """
        volatility = returns.rolling(window=lookback).std() * np.sqrt(252)
        return volatility

    @staticmethod
    def calculate_position_size(momentum: float,
                               volatility: float,
                               target_vol: float,
                               max_leverage: float,
                               min_weight: float,
                               max_weight: float) -> float:
        """
        Calculate position size using volatility scaling.

        Formula: weight = (target_vol / realized_vol) * sign(momentum) * |momentum|

        The position size is:
        1. Scaled by target_vol / realized_vol (inverse volatility weighting)
        2. Directional based on momentum sign
        3. Magnitude scaled by momentum strength
        4. Capped by leverage and concentration limits

        Args:
            momentum: Risk-adjusted momentum score
            volatility: Realized volatility of asset
            target_vol: Target portfolio volatility
            max_leverage: Maximum leverage allowed
            min_weight: Minimum position weight
            max_weight: Maximum position weight (concentration limit)

        Returns:
            Position weight (can be 0 if momentum is too weak)
        """
        if pd.isna(momentum) or pd.isna(volatility) or volatility <= 0:
            return 0.0

        # Calculate base weight using inverse volatility scaling
        # Higher volatility = lower weight for same momentum
        base_weight = (target_vol / volatility) * momentum

        # Apply leverage constraint
        base_weight = np.clip(base_weight, -max_leverage, max_leverage)

        # Apply concentration limits
        # We only go long in this strategy (positive momentum)
        if base_weight > 0:
            weight = np.clip(base_weight, min_weight, max_weight)
        else:
            weight = 0.0  # No short positions

        return weight


# ============================================================================
# BACKTEST ENGINE
# ============================================================================

class DynamicMomentumBacktest:
    """
    Main backtesting engine for Dynamic-Scaled Momentum strategy.

    Implements proper trading flow:
    1. Calculate signals at Close[t] using data up to t-1
    2. Execute trades at Open[t+1]
    3. Value portfolio at Open[t+1]
    4. Calculate returns from Open[t+1] to Open[t+2]
    5. During holding: Close[t] to Close[t+1]
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.data_fetcher = YahooDataFetcher()
        self.momentum_calc = MomentumCalculator()
        self.vol_scaler = VolatilityScaler()

        # Storage for results
        self.price_data: Dict[str, pd.DataFrame] = {}
        self.benchmark_data: Optional[pd.DataFrame] = None
        self.portfolio_value: pd.Series = None
        self.positions: pd.DataFrame = None
        self.trades: List[Dict] = []

    def fetch_universe(self) -> List[str]:
        """
        Fetch and filter trading universe.
        Only includes stocks with sufficient data and market cap.
        """
        print("Fetching S&P 500 universe...")
        tickers = self.data_fetcher.get_sp500_tickers()

        valid_tickers = []
        for ticker in tickers:
            # Fetch price data
            df = self.data_fetcher.get_historical_prices(
                ticker,
                self.config.START_DATE,
                self.config.END_DATE
            )

            # Filter: must have minimum trading days
            if len(df) >= self.config.MIN_TRADING_DAYS:
                # Check market cap (basic filter)
                # In production, you'd want to check historical market cap
                # For now, we'll just use stocks with sufficient price history
                valid_tickers.append(ticker)
                self.price_data[ticker] = df
                print(f"Added {ticker}: {len(df)} trading days")

        print(f"\nFinal universe: {len(valid_tickers)} stocks")
        return valid_tickers

    def fetch_benchmark(self):
        """Fetch benchmark data (URTH by default)."""
        print(f"\nFetching benchmark data: {self.config.BENCHMARK_TICKER}")
        self.benchmark_data = self.data_fetcher.get_historical_prices(
            self.config.BENCHMARK_TICKER,
            self.config.START_DATE,
            self.config.END_DATE
        )

        if self.benchmark_data.empty:
            print("Warning: Could not fetch benchmark data")
        else:
            print(f"Benchmark data: {len(self.benchmark_data)} days")

    def calculate_signals(self, date: pd.Timestamp, tickers: List[str]) -> pd.DataFrame:
        """
        Calculate momentum signals for all tickers at given date.

        Critical: Uses only data UP TO (not including) the signal date to avoid look-ahead bias.
        Signal calculated at Close[date], will be executed at Open[date+1].

        Returns DataFrame with columns: ticker, momentum, volatility, risk_adj_momentum, position_size
        """
        signals = []

        for ticker in tickers:
            df = self.price_data[ticker]

            # Get data up to signal date (not including future)
            historical_data = df[df.index < date]

            if len(historical_data) < max(self.config.MOMENTUM_LOOKBACK,
                                         self.config.VOLATILITY_LOOKBACK):
                continue

            # Calculate momentum using Close prices
            close_prices = historical_data['Close']
            momentum = self.momentum_calc.calculate_momentum(
                close_prices,
                self.config.MOMENTUM_LOOKBACK
            ).iloc[-1]

            # Calculate volatility using Close-to-Close returns
            returns = close_prices.pct_change()
            volatility = self.vol_scaler.calculate_volatility(
                returns,
                self.config.VOLATILITY_LOOKBACK
            ).iloc[-1]

            # Calculate risk-adjusted momentum
            risk_adj_momentum = self.momentum_calc.calculate_risk_adjusted_momentum(
                close_prices,
                self.config.MOMENTUM_LOOKBACK,
                self.config.VOLATILITY_LOOKBACK
            ).iloc[-1]

            # Skip if momentum is below threshold
            if momentum < self.config.MIN_MOMENTUM_THRESHOLD:
                continue

            # Calculate position size
            position_size = self.vol_scaler.calculate_position_size(
                risk_adj_momentum,
                volatility,
                self.config.TARGET_VOLATILITY,
                self.config.MAX_LEVERAGE,
                self.config.MIN_WEIGHT,
                self.config.MAX_WEIGHT
            )

            signals.append({
                'ticker': ticker,
                'momentum': momentum,
                'volatility': volatility,
                'risk_adj_momentum': risk_adj_momentum,
                'position_size': position_size
            })

        signals_df = pd.DataFrame(signals)

        if not signals_df.empty:
            # Sort by risk-adjusted momentum and select top N
            signals_df = signals_df.nlargest(self.config.TOP_N_STOCKS, 'risk_adj_momentum')

            # Normalize weights to sum to 1 (or less if max leverage not reached)
            total_weight = signals_df['position_size'].sum()
            if total_weight > 0:
                signals_df['position_size'] = signals_df['position_size'] / total_weight

        return signals_df

    def execute_trades(self,
                      current_date: pd.Timestamp,
                      signals_df: pd.DataFrame,
                      current_positions: Dict[str, float]) -> Tuple[Dict[str, float], float]:
        """
        Execute trades at Open[current_date] based on signals from Close[previous_date].

        Returns:
            - new_positions: Dictionary of ticker -> weight
            - transaction_costs: Total costs incurred from trading
        """
        new_positions = {}
        total_costs = 0.0

        # Get target positions from signals
        target_positions = {}
        for _, row in signals_df.iterrows():
            target_positions[row['ticker']] = row['position_size']

        # Calculate trades needed
        all_tickers = set(list(current_positions.keys()) + list(target_positions.keys()))

        for ticker in all_tickers:
            current_weight = current_positions.get(ticker, 0.0)
            target_weight = target_positions.get(ticker, 0.0)

            trade_size = abs(target_weight - current_weight)

            if trade_size > 1e-6:  # Only trade if meaningful change
                # Calculate transaction costs
                cost = trade_size * (self.config.COMMISSION_RATE + self.config.SLIPPAGE_RATE)
                total_costs += cost

                # Record trade - simplified logic
                # We only need to verify that the ticker exists and has data for this date
                if ticker in self.price_data and current_date in self.price_data[ticker].index:
                    self.trades.append({
                        'date': current_date,
                        'ticker': ticker,
                        'action': 'BUY' if target_weight > current_weight else 'SELL',
                        'from_weight': current_weight,
                        'to_weight': target_weight,
                        'trade_size': trade_size,
                        'price': self.price_data[ticker].loc[current_date, 'Open'],
                        'cost': cost
                    })

            # Update position
            if target_weight > 1e-6:
                new_positions[ticker] = target_weight

        return new_positions, total_costs

    def calculate_portfolio_return(self,
                                   positions: Dict[str, float],
                                   start_date: pd.Timestamp,
                                   end_date: pd.Timestamp,
                                   is_rebalance_day: bool) -> float:
        """
        Calculate portfolio return for period.

        Trading Flow:
        - On rebalance day: Open[t+1] to Open[t+2] (after executing trades)
        - During holding: Close[t] to Close[t+1] (no trades)

        Args:
            positions: Current portfolio positions (ticker -> weight)
            start_date: Start of return period
            end_date: End of return period
            is_rebalance_day: Whether this is a rebalance day

        Returns:
            Portfolio return for the period
        """
        if not positions:
            return 0.0

        portfolio_return = 0.0

        for ticker, weight in positions.items():
            if ticker not in self.price_data:
                continue

            df = self.price_data[ticker]

            # Ensure both dates exist in data
            if start_date not in df.index or end_date not in df.index:
                continue

            # Calculate return based on trading flow
            if is_rebalance_day:
                # Rebalance day: Use Open prices (execute at Open[t+1], value at Open[t+2])
                start_price = df.loc[start_date, 'Open']
                end_price = df.loc[end_date, 'Open']
            else:
                # Holding period: Use Close prices (Close[t] to Close[t+1])
                start_price = df.loc[start_date, 'Close']
                end_price = df.loc[end_date, 'Close']

            if pd.notna(start_price) and pd.notna(end_price) and start_price > 0:
                asset_return = (end_price - start_price) / start_price
                portfolio_return += weight * asset_return

        return portfolio_return

    def run_backtest(self):
        """
        Execute full backtest following the specified trading flow.
        """
        print("\n" + "="*80)
        print("STARTING DYNAMIC-SCALED MOMENTUM BACKTEST")
        print("="*80)

        # Fetch data
        tickers = self.fetch_universe()
        self.fetch_benchmark()

        if len(tickers) == 0:
            print("Error: No valid tickers in universe")
            return

        # Get common trading dates from all stocks
        all_dates = pd.DatetimeIndex([])
        for df in self.price_data.values():
            all_dates = all_dates.union(df.index)
        all_dates = sorted(all_dates)

        # Filter dates within backtest period
        all_dates = [d for d in all_dates if self.config.START_DATE <= d.strftime('%Y-%m-%d') <= self.config.END_DATE]

        print(f"\nBacktest period: {all_dates[0]} to {all_dates[-1]}")
        print(f"Total trading days: {len(all_dates)}")

        # Initialize backtest
        portfolio_values = [1.0]  # Start with $1
        portfolio_dates = [all_dates[0]]
        current_positions = {}
        days_since_rebalance = 0

        # Skip initial period to have enough data for momentum calculation
        start_idx = max(self.config.MOMENTUM_LOOKBACK, self.config.VOLATILITY_LOOKBACK) + 1

        print(f"\nStarting backtest from day {start_idx} to allow for momentum calculation...")

        # Main backtest loop
        for i in range(start_idx, len(all_dates)-1):
            current_date = all_dates[i]
            next_date = all_dates[i+1]

            # Check if it's time to rebalance
            is_rebalance_day = (days_since_rebalance >= self.config.REBALANCE_FREQUENCY)

            if is_rebalance_day:
                # Calculate signals at Close[current_date]
                signals = self.calculate_signals(current_date, tickers)

                # Execute trades at Open[next_date]
                new_positions, transaction_costs = self.execute_trades(
                    next_date,
                    signals,
                    current_positions
                )

                # Deduct transaction costs from portfolio value
                portfolio_values[-1] *= (1 - transaction_costs)

                # Update positions
                current_positions = new_positions
                days_since_rebalance = 0

                print(f"Rebalance on {next_date.strftime('%Y-%m-%d')}: "
                      f"{len(current_positions)} positions, "
                      f"costs: {transaction_costs:.4%}")

            # Calculate return for the period
            period_return = self.calculate_portfolio_return(
                current_positions,
                current_date,
                next_date,
                is_rebalance_day
            )

            # Update portfolio value
            new_value = portfolio_values[-1] * (1 + period_return)
            portfolio_values.append(new_value)
            portfolio_dates.append(next_date)

            days_since_rebalance += 1

            # Progress update
            if i % 50 == 0:
                print(f"Progress: {i}/{len(all_dates)} days, "
                      f"Portfolio value: ${new_value:.2f}")

        # Store results
        self.portfolio_value = pd.Series(portfolio_values, index=portfolio_dates)

        print("\n" + "="*80)
        print("BACKTEST COMPLETED")
        print("="*80)

    def calculate_performance_metrics(self) -> Dict:
        """
        Calculate comprehensive performance metrics for strategy and benchmark.
        """
        if self.portfolio_value is None:
            return {}

        # Strategy returns
        strategy_returns = self.portfolio_value.pct_change().dropna()

        # Benchmark returns (Close to Close during holding)
        if self.benchmark_data is not None and not self.benchmark_data.empty:
            # Align benchmark with strategy dates
            benchmark_aligned = self.benchmark_data.reindex(self.portfolio_value.index)
            benchmark_returns = benchmark_aligned['Close'].pct_change().dropna()
        else:
            benchmark_returns = pd.Series()

        metrics = {}

        # Strategy Metrics
        total_return = (self.portfolio_value.iloc[-1] / self.portfolio_value.iloc[0] - 1)
        ann_return = (1 + total_return) ** (252 / len(strategy_returns)) - 1
        ann_vol = strategy_returns.std() * np.sqrt(252)
        sharpe = ann_return / ann_vol if ann_vol > 0 else 0

        # Drawdown calculation
        cumulative = self.portfolio_value
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_drawdown = drawdown.min()

        metrics['Strategy'] = {
            'Total Return': total_return,
            'Annualized Return': ann_return,
            'Annualized Volatility': ann_vol,
            'Sharpe Ratio': sharpe,
            'Max Drawdown': max_drawdown,
            'Number of Trades': len(self.trades)
        }

        # Benchmark Metrics
        if not benchmark_returns.empty:
            bench_total_return = (1 + benchmark_returns).prod() - 1
            bench_ann_return = (1 + bench_total_return) ** (252 / len(benchmark_returns)) - 1
            bench_ann_vol = benchmark_returns.std() * np.sqrt(252)
            bench_sharpe = bench_ann_return / bench_ann_vol if bench_ann_vol > 0 else 0

            # Benchmark drawdown
            bench_cumulative = (1 + benchmark_returns).cumprod()
            bench_running_max = bench_cumulative.expanding().max()
            bench_drawdown = (bench_cumulative - bench_running_max) / bench_running_max
            bench_max_drawdown = bench_drawdown.min()

            metrics['Benchmark'] = {
                'Total Return': bench_total_return,
                'Annualized Return': bench_ann_return,
                'Annualized Volatility': bench_ann_vol,
                'Sharpe Ratio': bench_sharpe,
                'Max Drawdown': bench_max_drawdown
            }

        return metrics

    def plot_results(self):
        """
        Create comprehensive visualization of backtest results.
        """
        if self.portfolio_value is None:
            print("No results to plot")
            return

        # Set style
        plt.style.use('seaborn-v0_8-darkgrid')
        fig = plt.figure(figsize=(16, 12))

        # 1. Cumulative Returns
        ax1 = plt.subplot(3, 2, 1)
        ax1.plot(self.portfolio_value.index, self.portfolio_value.values,
                label='Strategy', linewidth=2, color='#2E86AB')

        if self.benchmark_data is not None and not self.benchmark_data.empty:
            benchmark_aligned = self.benchmark_data.reindex(self.portfolio_value.index)
            benchmark_cumulative = benchmark_aligned['Close'] / benchmark_aligned['Close'].iloc[0]
            ax1.plot(benchmark_cumulative.index, benchmark_cumulative.values,
                    label=f'Benchmark ({self.config.BENCHMARK_TICKER})',
                    linewidth=2, color='#A23B72', alpha=0.7)

        ax1.set_title('Cumulative Returns', fontsize=14, fontweight='bold')
        ax1.set_xlabel('Date')
        ax1.set_ylabel('Portfolio Value ($)')
        ax1.legend(loc='best')
        ax1.grid(True, alpha=0.3)

        # 2. Drawdown
        ax2 = plt.subplot(3, 2, 2)
        cumulative = self.portfolio_value
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        ax2.fill_between(drawdown.index, drawdown.values, 0,
                        color='#F18F01', alpha=0.6)
        ax2.set_title('Drawdown', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Date')
        ax2.set_ylabel('Drawdown (%)')
        ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
        ax2.grid(True, alpha=0.3)

        # 3. Rolling Returns (30-day)
        ax3 = plt.subplot(3, 2, 3)
        strategy_returns = self.portfolio_value.pct_change()
        rolling_returns = strategy_returns.rolling(30).mean() * 252  # Annualized
        ax3.plot(rolling_returns.index, rolling_returns.values,
                color='#2E86AB', linewidth=1.5)
        ax3.axhline(y=0, color='red', linestyle='--', alpha=0.5)
        ax3.set_title('Rolling 30-Day Returns (Annualized)', fontsize=14, fontweight='bold')
        ax3.set_xlabel('Date')
        ax3.set_ylabel('Return (%)')
        ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
        ax3.grid(True, alpha=0.3)

        # 4. Rolling Volatility (30-day)
        ax4 = plt.subplot(3, 2, 4)
        rolling_vol = strategy_returns.rolling(30).std() * np.sqrt(252)
        ax4.plot(rolling_vol.index, rolling_vol.values,
                color='#F18F01', linewidth=1.5)
        ax4.axhline(y=self.config.TARGET_VOLATILITY,
                   color='green', linestyle='--', alpha=0.5,
                   label=f'Target Vol ({self.config.TARGET_VOLATILITY:.0%})')
        ax4.set_title('Rolling 30-Day Volatility (Annualized)', fontsize=14, fontweight='bold')
        ax4.set_xlabel('Date')
        ax4.set_ylabel('Volatility (%)')
        ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
        ax4.legend(loc='best')
        ax4.grid(True, alpha=0.3)

        # 5. Monthly Returns Heatmap
        ax5 = plt.subplot(3, 2, 5)
        monthly_returns = strategy_returns.resample('ME').apply(lambda x: (1 + x).prod() - 1)
        monthly_returns_pivot = monthly_returns.to_frame('returns')
        monthly_returns_pivot['Year'] = monthly_returns_pivot.index.year
        monthly_returns_pivot['Month'] = monthly_returns_pivot.index.month

        if len(monthly_returns_pivot) > 0:
            heatmap_data = monthly_returns_pivot.pivot(index='Year',
                                                       columns='Month',
                                                       values='returns')
            sns.heatmap(heatmap_data, annot=True, fmt='.1%', cmap='RdYlGn',
                       center=0, cbar_kws={'label': 'Return'}, ax=ax5)
            ax5.set_title('Monthly Returns Heatmap', fontsize=14, fontweight='bold')
            ax5.set_xlabel('Month')
            ax5.set_ylabel('Year')

        # 6. Return Distribution
        ax6 = plt.subplot(3, 2, 6)
        ax6.hist(strategy_returns.dropna(), bins=50, color='#2E86AB',
                alpha=0.7, edgecolor='black')
        ax6.axvline(x=strategy_returns.mean(), color='red',
                   linestyle='--', linewidth=2, label=f'Mean: {strategy_returns.mean():.2%}')
        ax6.set_title('Daily Returns Distribution', fontsize=14, fontweight='bold')
        ax6.set_xlabel('Daily Return (%)')
        ax6.set_ylabel('Frequency')
        ax6.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
        ax6.legend(loc='best')
        ax6.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('dmom_backtest_results.png', dpi=300, bbox_inches='tight')
        print("\nResults plot saved as 'dmom_backtest_results.png'")
        plt.show()

    def print_summary(self):
        """Print detailed summary of backtest results."""
        print("\n" + "="*80)
        print("BACKTEST SUMMARY")
        print("="*80)

        metrics = self.calculate_performance_metrics()

        for strategy_name, strategy_metrics in metrics.items():
            print(f"\n{strategy_name} Performance:")
            print("-" * 40)
            for metric_name, value in strategy_metrics.items():
                if 'Return' in metric_name or 'Volatility' in metric_name or 'Drawdown' in metric_name:
                    print(f"{metric_name:.<30} {value:>10.2%}")
                elif 'Ratio' in metric_name:
                    print(f"{metric_name:.<30} {value:>10.2f}")
                else:
                    print(f"{metric_name:.<30} {value:>10}")

        # Trade Analysis
        if len(self.trades) > 0:
            print("\n" + "="*80)
            print("TRADE ANALYSIS")
            print("="*80)

            trades_df = pd.DataFrame(self.trades)

            # Basic statistics
            total_trades = len(trades_df)
            buys = len(trades_df[trades_df['action'] == 'BUY'])
            sells = len(trades_df[trades_df['action'] == 'SELL'])
            total_costs = trades_df['cost'].sum()
            avg_trade_size = trades_df['trade_size'].mean()

            print(f"\nTrading Statistics:")
            print(f"  Total Trades: {total_trades}")
            print(f"  Buy Orders: {buys} ({buys/total_trades:.1%})")
            print(f"  Sell Orders: {sells} ({sells/total_trades:.1%})")
            print(f"  Total Transaction Costs: {total_costs:.4%} of initial capital")
            print(f"  Average Trade Size: {avg_trade_size:.2%}")

            # Monthly turnover
            trades_df['date'] = pd.to_datetime(trades_df['date'])
            trades_df['month'] = trades_df['date'].dt.to_period('M')
            monthly_turnover = trades_df.groupby('month')['trade_size'].sum()

            print(f"\nTurnover Analysis:")
            print(f"  Average Monthly Turnover: {monthly_turnover.mean():.1%}")
            print(f"  Max Monthly Turnover: {monthly_turnover.max():.1%}")
            print(f"  Min Monthly Turnover: {monthly_turnover.min():.1%}")

            # Most traded stocks
            top_traded = trades_df['ticker'].value_counts().head(10)
            print(f"\nTop 10 Most Traded Stocks:")
            for ticker, count in top_traded.items():
                print(f"  {ticker}: {count} trades")

        print("\n" + "="*80)


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """
    Main execution function for Dynamic-Scaled Momentum strategy backtest.
    """
    # Initialize configuration
    config = StrategyConfig()

    # Print configuration
    print("\n" + "="*80)
    print("DYNAMIC-SCALED MOMENTUM (dMOM) STRATEGY")
    print("="*80)
    print(f"\nConfiguration:")
    print(f"  Backtest Period: {config.START_DATE} to {config.END_DATE}")
    print(f"  Momentum Lookback: {config.MOMENTUM_LOOKBACK} days")
    print(f"  Volatility Lookback: {config.VOLATILITY_LOOKBACK} days")
    print(f"  Rebalance Frequency: {config.REBALANCE_FREQUENCY} days")
    print(f"  Target Volatility: {config.TARGET_VOLATILITY:.1%}")
    print(f"  Max Leverage: {config.MAX_LEVERAGE}")
    print(f"  Top N Stocks: {config.TOP_N_STOCKS}")
    print(f"  Commission Rate: {config.COMMISSION_RATE:.2%}")
    print(f"  Slippage Rate: {config.SLIPPAGE_RATE:.2%}")
    print(f"  Benchmark: {config.BENCHMARK_TICKER}")

    # Initialize and run backtest
    backtest = DynamicMomentumBacktest(config)
    backtest.run_backtest()

    # Display results
    backtest.print_summary()
    backtest.plot_results()

    # Return backtest object for further analysis
    return backtest


if __name__ == "__main__":
    backtest = main()
